In [ ]:
%matplotlib inline
# Load the supplied table and plot for collecting the SMB outlet streams.
%run resources/advanced_chromatography.py

# 03.2 Advanced Chromatography

## Contents

**Tools:** CADET-Process (Python).

**Python used:** Model builders, event configuration, simulation and collecting outlet signals.

**Introduces:** valve routing, cyclic chromatography, `SerialZone`, `ParallelZone`, `CarouselBuilder` and collection over a full SMB cycle.

1. Valve dispersion
2. Simulated moving bed
3. Collected extract and raffinate

The RTD lesson separated instrument transport from column transport. We now include that instrument in a binding experiment, then connect several columns in a cyclic separation.

## Example 1: modeling dispersion of a valve

Mixers and tubing change the pulse before it enters the column and before it reaches the detector. Their delay and spreading must stay in the model when interpreting a binding experiment. This example adds the inlet valve; the exercises add tubing.

A `Cstr` represents mixing at the inlet valve, using the same model introduced in RTD.

Two `Inlet` units supply feed and protein-free eluent. Events switch their `flow_rate` values after loading.

```{figure} ./resources/valve_mixer.png
:width: 50%
:align: center
```

::::{grid} 1 1 2 2
:::{card}
:columns: 6

**Column transport settings:** geometry comes from the column; porosity and dispersion are estimates to characterize.

- length: $0.1~\text{m}$
- diameter: $0.01~\text{m}$
- bed porosity: $0.37$
- particle radius: $4.5 \times 10^{-5}~\text{m}$
- particle porosity: $0.33$
- axial dispersion: $2.0 \times 10^{-7}~\text{m}^2~\text{s}^{-1}$
:::

:::{card}
:columns: 6

**Langmuir binding settings for A and B:**

- adsorption rate: $[0.02, 0.03]~\text{m}^3~\text{mol}^{-1}~\text{s}^{-1}$
- desorption rate: $[1, 1]~\text{s}^{-1}$
- binding capacity: $[100, 100]~\text{mM}$
:::
::::

For the tank, assume a volume of $1~\text{mL}$.
Moreover, consider a flow rate of $1~\text{mL}~\text{min}^{-1}$, a feed concentration of $10~\text{mM}$, and a feed duration of $60~\text{s}$.

In [ ]:
# Supplied model: Langmuir column fed through a mixing valve from two inlets.
Q = 1e-6/60

from CADETProcess.processModel import ComponentSystem
from CADETProcess.processModel import Langmuir
from CADETProcess.processModel import (
    Inlet, Cstr, LumpedRateModelWithPores, Outlet
)
from CADETProcess.processModel import FlowSheet
from CADETProcess.processModel import Process

# Component System
component_system = ComponentSystem(['A', 'B'])

# Binding Model
binding_model = Langmuir(component_system, name='langmuir')
binding_model.is_kinetic = False
binding_model.adsorption_rate = [0.02, 0.03]
binding_model.desorption_rate = [1, 1]
binding_model.capacity = [100, 100]

# Unit Operations
feed = Inlet(component_system, name='feed')
feed.c = [10, 10]

eluent = Inlet(component_system, name='eluent')
eluent.c = [0, 0]

# Mixer
valve = Cstr(component_system, 'valve')
valve.init_liquid_volume = 1e-6
valve.const_solid_volume = 0

# Column
column = LumpedRateModelWithPores(component_system, 'column')
column.binding_model = binding_model
column.length = 0.1
column.diameter = 0.01
column.bed_porosity = 0.37
column.particle_radius = 4.5e-5
column.particle_porosity = 0.33
column.axial_dispersion = 2.0e-7
column.film_diffusion = [1e-4, 1e-4]

outlet = Outlet(component_system, name='outlet')

# Flow Sheet
flow_sheet = FlowSheet(component_system)

flow_sheet.add_unit(feed)
flow_sheet.add_unit(eluent)
flow_sheet.add_unit(valve)
flow_sheet.add_unit(column)
flow_sheet.add_unit(outlet)

flow_sheet.add_connection(feed, valve)
flow_sheet.add_connection(eluent, valve)
flow_sheet.add_connection(valve, column)
flow_sheet.add_connection(column, outlet)

# Process
process = Process(flow_sheet, 'batch elution')
process.cycle_time = 1200

The two inlets take turns: feed flows for the first **60 s**, then eluent flows until the end. Four events switch the inlets' `flow_rate` between `Q` and `0`:

| Event | Parameter path | Value | Time / s |
|---|---|---|---:|
| `feed_on` | `flow_sheet.feed.flow_rate` | `Q` | 0 |
| `feed_off` | `flow_sheet.feed.flow_rate` | `0` | 60 |
| `eluent_off` | `flow_sheet.eluent.flow_rate` | `0` | 0 |
| `eluent_on` | `flow_sheet.eluent.flow_rate` | `Q` | 60 |

The total flow into the valve stays at `Q`; only its source changes.

In [ ]:
# Flow-rate events: feed first, then eluent
process.add_event('feed_on', 'flow_sheet.feed.flow_rate', Q, 0)
process.add_event('feed_off', 'flow_sheet.feed.flow_rate', 0.0, 60)

process.add_event('eluent_off', 'flow_sheet.eluent.flow_rate', 0.0, 0)
process.add_event('eluent_on', 'flow_sheet.eluent.flow_rate', Q, 60)

The simulation shows the column inlet and outlet signals side by side. The inlet already includes the mixer response.

In [ ]:
from CADETProcess.simulator import Cadet
process_simulator = Cadet()

simulation_results = process_simulator.simulate(process)
_ = simulation_results.solution.column.inlet.plot()
_ = simulation_results.solution.column.outlet.plot()

## Simulated moving bed (cyclic systems)

A **simulated moving bed (SMB)** uses several columns to separate a continuous feed into extract and raffinate. Switching connections moves each column through zones with different roles.

The model uses a carousel to represent the changing connections. In a laboratory SMB, valves switch the connections while the columns usually stay in place.

::::{grid} 1 1 2 2

:::{grid-item}
:columns: 6

For example, in a classical SMB, four zones are present:

- Zone I: Elution of the strongly adsorbing component
- Zone II: Elution of the weakly adsorbing component
- Zone III: Adsorption of the strongly adsorbing component
- Zone IV: Adsorption of the weakly adsorbing component
:::

:::{grid-item}
:columns: 6

```{figure} ./resources/smb_zones.svg
:width: 100%
:align: center
:alt: Four zones in a closed loop. Eluent enters before zone I, extract leaves after zone I, feed enters before zone III, raffinate leaves after zone III. The liquid flows clockwise from zone I to IV.
```
:::
::::

Moreover, four in- and outlets are connected to the zones:

- Feed: Inlet containing the components to be separated
- Eluent: Inlet with elution buffer
- Extract: Outlet containing the strongly adsorbing component
- Raffinate: Outlet containing the weakly adsorbing component

`CarouselBuilder` assembles the columns, zone connections and switching events into a `Process`.

### Zones, columns and switching

- A **zone** is a position in the loop between two connections, named by its role. Zone numbers are not column numbers. In this example each zone holds one column; a zone can also hold several.
- The liquid flows around a **closed loop**, I → II → III → IV → I. Eluent and feed join the loop before zones I and III; extract and raffinate leave it after zones I and III. That gives two inlets and two outlets.
- After every `switch_time`, each column moves one zone back against the liquid flow, from zone I to zone IV, then III and II. The columns thus carry the bound components against the liquid, as a moving solid would.
- `SerialZone` and `ParallelZone` describe the columns **within one zone**: one after another, or side by side with the flow split between them.

### Define the column template

Binding and transport are configured once. `CarouselBuilder` uses this column as the template for each position.

In [ ]:
from CADETProcess.processModel import ComponentSystem
component_system = ComponentSystem(["Protein A", "Protein B"])

# Binding Model
from CADETProcess.processModel import Linear
binding_model = Linear(component_system)
binding_model.adsorption_rate = [6, 8]
binding_model.desorption_rate = [1, 1]

from CADETProcess.processModel import LumpedRateModelWithoutPores
column = LumpedRateModelWithoutPores(component_system, name='master_column')
column.length = 0.6
column.diameter = 0.024
column.axial_dispersion = 4.7e-7
column.total_porosity = 0.7

column.binding_model = binding_model

### Define the streams

Feed and eluent enter the system; extract and raffinate collect the separated components.

In [ ]:
from CADETProcess.processModel import Inlet, Outlet
feed = Inlet(component_system, name='feed')
feed.c = [10, 10]
feed.flow_rate = 2e-7
eluent = Inlet(component_system, name='eluent')
eluent.c = [0, 0]
eluent.flow_rate = 6e-7

raffinate = Outlet(component_system, name='raffinate')
extract = Outlet(component_system, name='extract')

### Define the zones

`SerialZone` connects columns in series; `ParallelZone` distributes flow across parallel columns. Each zone takes the component system, a name and its number of columns.

**Task:** create four serial zones, `zone_I` to `zone_IV`, each with **one column**.

In [ ]:
# Four serial zones
from CADETProcess.modelBuilder import SerialZone, ParallelZone

zone_I = SerialZone(component_system, 'zone_I', 1)
zone_II = SerialZone(component_system, 'zone_II', 1)
zone_III = SerialZone(component_system, 'zone_III', 1)
zone_IV = SerialZone(component_system, 'zone_IV', 1)

Assign the column template and add streams and zones to `CarouselBuilder`, as units are added to a `FlowSheet`.

In [ ]:
from CADETProcess.modelBuilder import CarouselBuilder

builder = CarouselBuilder(component_system, 'smb')
builder.column = column
builder.add_unit(feed)
builder.add_unit(eluent)

builder.add_unit(raffinate)
builder.add_unit(extract)

builder.add_unit(zone_I)
builder.add_unit(zone_II)
builder.add_unit(zone_III)
builder.add_unit(zone_IV)

### Connect the zones

The connections are added in flow order. `set_output_state()` sets the split between outgoing connections, in the order they were added.

**Task:** connect streams and zones as in the SMB scheme:

| From | To | Split of the outgoing flow |
|---|---|---|
| `eluent` | `zone_I` | |
| `zone_I` | `extract`, then `zone_II` | `[0.15, 0.85]` |
| `zone_II` | `zone_III` | |
| `feed` | `zone_III` | |
| `zone_III` | `raffinate`, then `zone_IV` | `[0.15, 0.85]` |
| `zone_IV` | `zone_I` | |

In [ ]:
# Connect streams and zones
builder.add_connection(eluent, zone_I)

builder.add_connection(zone_I, extract)
builder.add_connection(zone_I, zone_II)
w_e = 0.15
builder.set_output_state(zone_I, [w_e, 1-w_e])

builder.add_connection(zone_II, zone_III)

builder.add_connection(feed, zone_III)

builder.add_connection(zone_III, raffinate)
builder.add_connection(zone_III, zone_IV)
w_r = 0.15
builder.set_output_state(zone_III, [w_r, 1-w_r])

builder.add_connection(zone_IV, zone_I)

### Build and simulate the process

With the switching interval set, `build_process()` returns a `Process` with the expanded flow sheet and switching events, which `Cadet` simulates.

In [ ]:
builder.switch_time = 300
process = builder.build_process()

**Three full cycles** show the startup of the SMB. The column states change between cycles; three cycles do not establish cyclic stationarity. The supplied simulation cell can take a little time to run.

In [ ]:
# Simulate the configured SMB for three complete startup cycles.
from CADETProcess.simulator import Cadet

process_simulator = Cadet()
process_simulator.n_cycles = 3
simulation_results = process_simulator.simulate(process)

Protein A binds more weakly (`adsorption_rate=6`); Protein B binds more strongly (`8`). A separating SMB should enrich **A in the raffinate** and **B in the extract**. The two outlet signals show where each protein leaves.

In [ ]:
# Shade the final cycle: this is the collection interval used in the next step.
collection_start = (simulation_results.n_cycles - 1) * process.cycle_time / 60
collection_end = simulation_results.n_cycles * process.cycle_time / 60
fig, ax = simulation_results.solution.raffinate.inlet.plot()
ax.set_title("Raffinate: target Protein A")
ax.axvspan(collection_start, collection_end, color="gray", alpha=0.15, label="Collected cycle")
ax.legend()
fig, ax = simulation_results.solution.extract.inlet.plot()
ax.set_title("Extract: target Protein B")
ax.axvspan(collection_start, collection_end, color="gray", alpha=0.15, label="Collected cycle")
ax.legend()
plt.show()

## What is collected?

Collect each outlet continuously over the **last simulated cycle**, shaded in the concentration plots (**40–60 min** for this run). CADET-Process's `create_fraction()` integrates concentration times flow rate over the interval,

$$n_i=\int c_i(t)Q(t)\,dt,$$

and also calculates the liquid volume. Our molar concentration units give amounts in **mol**, displayed here as **µmol**. This compares the collected pools, rather than the composition at a single time point.

In [ ]:
# Pool the raffinate, extract and feed over the last complete cycle.
raffinate_signal = simulation_results.solution_cycles.raffinate.inlet[-1]
extract_signal = simulation_results.solution_cycles.extract.inlet[-1]
feed_signal = simulation_results.solution_cycles.feed.outlet[-1]

raffinate_pool = raffinate_signal.create_fraction()
extract_pool = extract_signal.create_fraction()
feed_pool = feed_signal.create_fraction()
_ = show_smb_collection(raffinate_pool, extract_pool, feed_pool)

For the supplied settings, the raffinate is about **99.8% Protein A** and the extract about **98.9% Protein B**, on a **molar basis**. There is separation, and both streams still contain a small amount of the other protein.

```{note}
High product composition does not establish high recovery. In this startup cycle, the total amounts leaving through both outlets are smaller than the feed input: the amount retained in the system is still increasing. Steady-state performance requires simulating to cyclic stationarity.
```

## Summary

- Valve and tubing transport remain part of the binding experiment.
- `CarouselBuilder` expands column templates, zones and switching settings into a process.
- Integrate outlet concentration times flow to compare collected products; startup cycles do not establish cyclic stationarity.

**Now:** [**03.2 Advanced Chromatography: Exercises**](03.2b_advanced_chromatography_exercises.ipynb): add tubing to the valve model and build a bind–elute carousel.

**Next lesson:** [**04.1 Optimization Essentials**](../../04%20Characterization%20and%20Optimization/04.1%20Optimization%20Essentials/04.1a_optimization_essentials_lesson.ipynb), the introduction to chapter 04, gives the background on fitting and process-design objectives.